In [ ]:
"""
judge_batch_v2.py  -  batch LLM-as-judge (3 metrics, LOCAL Ollama judges) over a ZIP of CSV files.

What it does when you run it
  1. asks you for the ZIP (Colab upload dialog) - or uses ZIP_PATH if you set it
  2. extracts it, finds EVERY .csv inside (any depth), and maps the columns automatically:
        question : question | query | prompt            (first one found, in this order)
        answer   : answer | response
        docs     : top_k_docs | retrieved_docs | ...      (chunks separated by |||)
     a column-mapping table is printed and saved; a file whose columns cannot be mapped is reported and skipped
  3. judges every file with 3 metrics:   ctx_answer, ctx_question, answer_question
  4. every judge's own score is saved separately:   gemma__ctx_answer, llama__ctx_answer, ...   (+ the aggregate)
  5. saves everything in a Google Drive folder (mirrors the folder structure of the zip):
        OUT_DIR/<same sub-folders>/<file>__judged.csv     one result file per input file
        OUT_DIR/_manifest.csv            what happened to each file (column mapping, rows, failures, time)
        OUT_DIR/_column_mapping.csv      which original column was used as question / answer / docs
        OUT_DIR/_debug_failures.jsonl    every unparsable / failed judge reply (raw text kept)
        OUT_DIR/_ALL_wide.csv  _ALL_long.csv  _summary.csv      all files together
  The run is resumable: finished rows / files are skipped when you run it again.

Colab:
    !ollama pull gemma3 ; !ollama pull llama3.1        # names must match JUDGES below (see `ollama list`)
    %run -i /content/judge_batch_v2.py
Optional, define BEFORE %run -i :
    LIMIT_FILES = 1 ; LIMIT_ROWS = 2        # quick test on one file, two rows
    ZIP_PATH = "/content/drive/MyDrive/xxx.zip"
    CONTEXT_MODE = "per_chunk"              # default "all" (all chunks in one prompt)
"""
import os, re, io, json, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from langchain_ollama import OllamaLLM
from tqdm.auto import tqdm

# ----------------------------------------------------------------------------- config
JUDGES = {"deepseek": "deepseek-r1", "llama": "llama3.2"}   # label (used in column names) -> LOCAL Ollama model name
WEIGHTS = None                                       # e.g. {"gemma": 0.6, "llama": 0.4}; None = equal
OLLAMA_URL = "http://localhost:11434"
NUM_CTX = 16384                                      # prompts reach ~10k tokens (4 chunks + long answer)
TEMPERATURE = 0.0
REPEATS = 1                                          # >1 only useful with TEMPERATURE > 0
PARALLEL = False                                     # True = ask both judges at once (needs GPU memory for both)
CONTEXT_MODE = globals().get("CONTEXT_MODE", "all")  # "all" | "per_chunk"
CHUNK_SEP = "|||"
EXPECTED_CHUNKS = 4

ZIP_PATH = globals().get("ZIP_PATH")                 # None -> upload dialog
EXTRACT_DIR = globals().get("EXTRACT_DIR", "/content/judge_input")
OUT_DIR = globals().get("OUT_DIR", "/content/drive/MyDrive/judge_results_3metrics")
LIMIT_FILES = globals().get("LIMIT_FILES")
LIMIT_ROWS = globals().get("LIMIT_ROWS")

# accepted column names (compared lower-case, spaces -> "_"); first match in each list wins
ALIASES = {
    "question": ["question", "query", "prompt", "questions"],
    "answer":   ["answer", "response", "answers", "generated_answer"],
    "docs":     ["top_k_docs", "retrieved_docs", "retrieved_context", "retrieved_contexts", "contexts", "context", "docs"],
}

METRICS = ("ctx_answer", "ctx_question", "answer_question")
META_COLS = ["source_file", "combo", "generator", "embedder", "domain", "variation"]

# ----------------------------------------------------------------------------- prompts (texts unchanged from your code)
PROMPT_CTX_QUESTION_ALL = """Role: You are a Context Relevance Expert evaluating retrieval quality.
Task: Assess how well the retrieved context supports answering the user's question.
Criteria: Judge the degree to which the information in the retrieved context is relevant, factual, and sufficient to produce a complete and accurate answer to the question.

Question: {question}
Retrieved Context: {contexts}

Output Format: Provide a score from 0.0 (Irrelevant/Useless) to 1.0 (Highly relevant and sufficient). Output only the number.
"""

PROMPT_CTX_ANSWER_ALL = """Role: You are a Grounding Analyst evaluating factual support.
Task: Assess how well the generated answer is grounded in the retrieved context.
Criteria: Judge the degree to which all claims in the answer are logically and factually supported by the entire retrieved context.

Retrieved Context: {contexts}
Generated Answer: {answer}

Output Format: Provide a score from 0.0 (Ungrounded/Hallucinated) to 1.0 (Fully supported). Output only the number.
"""

PROMPT_CTX_QUESTION_CHUNK = """Role: You are a Chunk Relevance Assessor.
Task: Evaluate the individual relevance of the chunk to the user's question.
Criteria: Judge how essential the information in the chunk is for producing a correct and complete answer to the question.

Question: {question}
Retrieved Chunk: {chunk}

Output Format: Provide a score from 0.0 (Irrelevant) to 1.0 (Highly essential). Output only the number.
"""

PROMPT_CTX_ANSWER_CHUNK = """Role: You are a Context Utility Validator.
Task: Determine whether the retrieved chunk contributed meaningfully to the generated answer.
Criteria: Score based on explicit factual or semantic overlap between the chunk and the answer.

Retrieved Chunk: {chunk}
Generated Answer: {answer}

Output Format: Provide a score from 0.0 (No overlap/Not used) to 1.0 (Directly relevant and reflected in the answer). Output only the number.
"""

PROMPT_ANSWER_QUESTION = """Role: You are an Answer Quality Judge evaluating relevance.
Task: Assess whether the generated answer directly and meaningfully addresses the question.
Criteria: A high score means the answer accurately, directly, and thoroughly responds to the core intent of the question.

Question: {question}
Generated Answer: {answer}

Output Format: Provide a score from 0.0 (Completely off-topic) to 1.0 (Perfectly addresses the question). Output only the number.
"""


# ----------------------------------------------------------------------------- parsing
_THINK = re.compile(r"<think>.*?</think>", re.S)
_NUM = re.compile(r"(?<![\d.])(\d+(?:\.\d+)?|\.\d+)")


def parse_score(text):
    """First number in the reply; must be in [0, 1]. Anything else (empty, '8/10', no number) -> None."""
    if not text:
        return None
    m = _NUM.search(_THINK.sub("", str(text)).strip())
    if not m:
        return None
    v = float(m.group(1))
    return v if 0.0 <= v <= 1.0 else None


def parse_chunks(cell):
    return [c.strip() for c in str(cell).split(CHUNK_SEP) if c.strip()]


# ----------------------------------------------------------------------------- judge
class Judge:
    def __init__(self, judges, weights=None, num_ctx=NUM_CTX, temperature=TEMPERATURE,
                 repeats=REPEATS, parallel=PARALLEL, base_url=OLLAMA_URL):
        assert len(set(judges.values())) == len(judges), "use different models for different judges"
        self.labels, self.model_of = list(judges), dict(judges)
        self.repeats, self.num_ctx, self.parallel = repeats, num_ctx, parallel
        w = np.array([1.0 if weights is None else float(weights[l]) for l in self.labels])
        self.w = w / w.sum()
        self.llms = {l: OllamaLLM(model=judges[l], base_url=base_url, num_ctx=num_ctx,
                                  temperature=temperature, keep_alive="30m") for l in self.labels}
        self.n_calls = {l: 0 for l in self.labels}
        self.n_failed = {l: 0 for l in self.labels}
        self.failures = []          # raw replies that could not be parsed (debug)
        self.current = {}           # file / row currently being judged (copied into failure records)
        self._warned_len = False

    def _ask(self, label, prompt, tag):
        raw, err = None, None
        for attempt in range(3):                              # retry only real errors (connection, model busy)
            try:
                raw, err = str(self.llms[label].invoke(prompt)), None
                break
            except Exception as e:
                err = repr(e)
                time.sleep(2 * (attempt + 1))
        score = parse_score(raw) if raw is not None else None
        if score is None:
            self.failures.append({**self.current, "tag": tag, "judge": label, "model": self.model_of[label],
                                  "prompt_chars": len(prompt), "raw_reply": (raw or "")[:300], "error": err})
        return score

    def score(self, prompt, tag=""):
        """-> (aggregate score or None, {judge label: score or None})"""
        if not self._warned_len and len(prompt) > 3.0 * self.num_ctx:
            print(f"  WARNING: prompt has {len(prompt)} chars, may exceed num_ctx={self.num_ctx}; Ollama truncates silently.")
            self._warned_len = True
        workers = len(self.labels) * self.repeats if self.parallel else 1
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = {l: [ex.submit(self._ask, l, prompt, tag) for _ in range(self.repeats)] for l in self.labels}
            per = {}
            for l, fl in futs.items():
                vals = [f.result() for f in fl]
                self.n_calls[l] += len(vals)
                self.n_failed[l] += sum(v is None for v in vals)
                vals = [v for v in vals if v is not None]
                per[l] = float(np.mean(vals)) if vals else None
        ok = [(i, per[l]) for i, l in enumerate(self.labels) if per[l] is not None]
        if not ok:
            return None, per
        w = np.array([self.w[i] for i, _ in ok])
        s = np.array([v for _, v in ok])
        return float((w * s).sum() / w.sum()), per


# ----------------------------------------------------------------------------- the three metrics
def _mean_or_none(xs):
    xs = [x for x in xs if x is not None]
    return float(np.mean(xs)) if xs else None


def score_context(judge, kind, text, chunks, mode=CONTEXT_MODE):
    """kind = 'question' or 'answer' (what the chunks are compared with)."""
    assert kind in ("question", "answer") and mode in ("all", "per_chunk")
    tag = f"ctx_{kind}"
    if mode == "all":
        tmpl = PROMPT_CTX_QUESTION_ALL if kind == "question" else PROMPT_CTX_ANSWER_ALL
        score, per = judge.score(tmpl.format(contexts="\n\n".join(chunks), **{kind: text}), tag)
        return {"score": score, "per_judge": per}
    tmpl = PROMPT_CTX_QUESTION_CHUNK if kind == "question" else PROMPT_CTX_ANSWER_CHUNK
    res = [judge.score(tmpl.format(chunk=c, **{kind: text}), f"{tag}[chunk{i}]") for i, c in enumerate(chunks, 1)]
    return {"score": _mean_or_none([s for s, _ in res]),
            "per_judge": {l: _mean_or_none([p[l] for _, p in res]) for l in judge.labels},
            "per_chunk": [s for s, _ in res]}


def score_answer_question(judge, question, answer):
    score, per = judge.score(PROMPT_ANSWER_QUESTION.format(question=question, answer=answer), "answer_question")
    return {"score": score, "per_judge": per}


def evaluate_row(judge, question, answer, chunks, mode=CONTEXT_MODE):
    return {"ctx_answer": score_context(judge, "answer", answer, chunks, mode),
            "ctx_question": score_context(judge, "question", question, chunks, mode),
            "answer_question": score_answer_question(judge, question, answer)}


def flatten(res, judge):
    """aggregate columns first, then ALL columns of judge 1, ALL columns of judge 2, ..., then disagreement."""
    rec = {k: res[k]["score"] for k in METRICS}
    for l in judge.labels:
        for k in METRICS:
            rec[f"{l}__{k}"] = res[k]["per_judge"][l]
    for k in METRICS:
        v = [res[k]["per_judge"][l] for l in judge.labels if res[k]["per_judge"][l] is not None]
        rec[f"{k}__judge_gap"] = (max(v) - min(v)) if len(v) > 1 else None      # |max - min| between judges
    if "per_chunk" in res["ctx_answer"]:
        for k in ("ctx_answer", "ctx_question"):
            rec[f"{k}__per_chunk"] = ";".join("" if s is None else f"{s:.3f}" for s in res[k]["per_chunk"])
    return rec


# ----------------------------------------------------------------------------- input: zip, files, columns
class ColumnMismatch(Exception):
    pass


def ensure_drive():
    if OUT_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")


def get_zip():
    if ZIP_PATH:
        assert os.path.exists(ZIP_PATH), f"ZIP_PATH not found: {ZIP_PATH}"
        return ZIP_PATH
    from google.colab import files
    print("Select the ZIP file with all CSVs ...")
    up = files.upload()
    zips = [n for n in up if n.lower().endswith(".zip")]
    assert zips, "no .zip was uploaded"
    return zips[0]


def extract_zip(zip_path, dest=EXTRACT_DIR):
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    csvs = sorted(str(p) for p in Path(dest).rglob("*.csv")
                  if "__MACOSX" not in p.parts and not p.name.startswith("._") and not p.name.startswith("."))
    assert csvs, "no CSV found inside the zip"
    root = os.path.commonpath([os.path.dirname(p) for p in csvs]) if len(csvs) > 1 else os.path.dirname(csvs[0])
    return [(p, os.path.relpath(p, root)) for p in csvs]


def map_columns(cols):
    norm = {re.sub(r"\s+", "_", str(c).strip().lower()): c for c in cols}
    found, notes = {}, []
    for std, cands in ALIASES.items():
        hits = [c for c in cands if c in norm]
        if not hits:
            raise ColumnMismatch(f"no column for '{std}' (accepted: {cands}); columns are {list(cols)}")
        found[std] = norm[hits[0]]
        if len(hits) > 1:
            notes.append(f"{std}: used '{norm[hits[0]]}', ignored {[norm[h] for h in hits[1:]]}")
    return found, "; ".join(notes)


def read_csv_strict(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")                       # strict: never silently drops lines
    except Exception as e:
        print(f"  strict read failed ({type(e).__name__}); retrying with python engine, bad lines reported")
        return pd.read_csv(path, encoding="utf-8-sig", engine="python", on_bad_lines="warn")


def load_standardized(path):
    raw = read_csv_strict(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    mp, note = map_columns(raw.columns)
    df = pd.DataFrame({"row_id": range(len(raw)),
                       "question": raw[mp["question"]], "answer": raw[mp["answer"]], "docs": raw[mp["docs"]]})
    if "batch_id" in raw.columns:
        df["batch_id"] = raw["batch_id"]
    for c in ("question", "answer", "docs"):
        df[c] = df[c].where(df[c].notna(), "").astype(str).str.strip()
    n_in = len(df)
    df = df[(df.question != "") & (df.answer != "") & (df.docs != "")]
    return df, mp, note, n_in


def parse_meta(rel):
    parts = list(Path(rel).parts)
    folders = [p.lower() for p in parts[:-1]]
    allp = [p.lower() for p in parts]

    def pick(words, pool):
        for w in words:
            if any(w in p for p in pool):
                return w
        return ""
    combo = re.sub(r"\s+", "", parts[0]).lower() if len(parts) > 1 else ""
    domain = (pick(["ecolog"], folders) or pick(["ecolog"], allp)) and "ecological" or \
             ((pick(["social"], folders) or pick(["social"], allp)) and "social" or "")
    variation = (pick(["context"], folders) and "context") or (pick(["prompt"], folders) and "prompt") or \
                (pick(["context"], allp) and "context") or (pick(["prompt"], allp) and "prompt") or ""
    gen, _, emb = combo.partition("-")
    return {"source_file": rel, "combo": combo, "generator": gen, "embedder": emb,
            "domain": domain, "variation": variation}


# ----------------------------------------------------------------------------- run one file / all files
def out_path_for(rel):
    p = Path(rel)
    return os.path.join(OUT_DIR, str(p.parent), p.stem + "__judged.csv")


def process_file(judge, path, rel, mode=CONTEXT_MODE, limit_rows=None):
    meta = parse_meta(rel)
    df, mp, note, n_in = load_standardized(path)
    out_path = out_path_for(rel)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if limit_rows:
        df = df.head(limit_rows)

    rows, done = [], set()
    if os.path.exists(out_path):
        old = pd.read_csv(out_path)
        assert "row_id" in old.columns, f"{out_path} was not written by this script"
        rows, done = old.to_dict("records"), set(old["row_id"])
    todo = df[~df.row_id.isin(done)]
    t_file = time.time()
    n_fail0 = sum(judge.n_failed.values())
    for _, r in tqdm(todo.iterrows(), total=len(todo), desc=Path(rel).stem[:40], leave=False):
        chunks = parse_chunks(r["docs"])
        judge.current = {"source_file": rel, "row_id": int(r["row_id"])}
        t0 = time.time()
        res = evaluate_row(judge, r["question"], r["answer"], chunks, mode)
        rec = {**meta, "row_id": int(r["row_id"])}
        if "batch_id" in df.columns:
            rec["batch_id"] = r["batch_id"]
        rec.update({"n_chunks": len(chunks), "context_mode": mode, "seconds": round(time.time() - t0, 1),
                    **flatten(res, judge), "question": r["question"], "answer": r["answer"]})
        rows.append(rec)
        pd.DataFrame(rows).to_csv(out_path, index=False)          # saved after every row

    if judge.failures:
        with open(os.path.join(OUT_DIR, "_debug_failures.jsonl"), "a", encoding="utf-8") as f:
            for x in judge.failures:
                f.write(json.dumps(x, ensure_ascii=False) + "\n")
        judge.failures = []
    out = pd.DataFrame(rows)
    return {**meta,
            "col_question": mp["question"], "col_answer": mp["answer"], "col_docs": mp["docs"], "mapping_note": note,
            "rows_in_csv": n_in, "rows_valid": len(df), "rows_judged": len(out), "rows_new_this_run": len(todo),
            "chunks_min": int(out.n_chunks.min()) if len(out) else None,
            "chunks_max": int(out.n_chunks.max()) if len(out) else None,
            "failed_calls": sum(judge.n_failed.values()) - n_fail0, "seconds": round(time.time() - t_file, 1),
            "status": "ok", "output": out_path}


def build_outputs(judge):
    files = sorted(Path(OUT_DIR).rglob("*__judged.csv"))
    if not files:
        return
    wide = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    wide.to_csv(os.path.join(OUT_DIR, "_ALL_wide.csv"), index=False)

    ids = META_COLS + ["row_id"]
    parts = []
    for k in METRICS:
        for j in ["aggregate"] + judge.labels:
            col = k if j == "aggregate" else f"{j}__{k}"
            if col in wide.columns:
                s = wide[ids + [col]].rename(columns={col: "score"})
                s["metric"], s["judge"] = k, j
                parts.append(s)
    pd.concat(parts, ignore_index=True).to_csv(os.path.join(OUT_DIR, "_ALL_long.csv"), index=False)

    score_cols = [c for c in wide.columns if c in METRICS or any(c == f"{l}__{k}" for l in judge.labels for k in METRICS)]
    summ = wide.groupby(META_COLS)[score_cols].agg("mean").round(4)
    summ.insert(0, "n_rows", wide.groupby(META_COLS).size())
    summ.reset_index().to_csv(os.path.join(OUT_DIR, "_summary.csv"), index=False)
    print(f"all-files outputs written ({len(files)} files, {len(wide)} rows) -> {OUT_DIR}")


def smoke_test(judge):
    """Tiny example: checks every judge is reachable and returns a parsable number."""
    q = "How does consumer demand contribute to modern slavery?"
    a = "Consumers create demand for goods produced through exploitation, even unknowingly, through price-driven choices."
    chunks = ["Consumers spend money on goods produced with the input of persons trapped in exploitation, "
              "thus knowingly or unknowingly feeding exploitation through their spending.",
              "Complex global supply chains mean that modern slavery can be less visible to the public eye."]
    res = evaluate_row(judge, q, a, chunks, "all")
    ok = True
    for k, v in res.items():
        print(f"  {k:16s} aggregate={v['score']}  per judge={ {l: (None if s is None else round(s, 3)) for l, s in v['per_judge'].items()} }")
    for l in judge.labels:
        if all(res[k]["per_judge"][l] is None for k in METRICS):
            ok = False
            print(f"  !! judge '{l}' ({judge.model_of[l]}) returned no valid number: is the model pulled? (`ollama list`)")
    judge.failures = []
    return ok


def run_all(judge):
    ensure_drive()
    os.makedirs(OUT_DIR, exist_ok=True)
    zip_path = get_zip()
    files = extract_zip(zip_path)
    if LIMIT_FILES:
        files = files[:LIMIT_FILES]
    print(f"\n{len(files)} CSV files found in the zip.\n")

    # 1) column check for ALL files before spending any GPU time
    plan, bad = [], []
    for path, rel in files:
        try:
            df, mp, note, n_in = load_standardized(path)
            plan.append({**parse_meta(rel), "col_question": mp["question"], "col_answer": mp["answer"],
                         "col_docs": mp["docs"], "rows_in_csv": n_in, "rows_valid": len(df), "note": note})
        except ColumnMismatch as e:
            bad.append({"source_file": rel, "status": "SKIPPED (columns)", "error": str(e)})
    plan_df = pd.DataFrame(plan)
    plan_df.to_csv(os.path.join(OUT_DIR, "_column_mapping.csv"), index=False)
    with pd.option_context("display.width", 250, "display.max_colwidth", 60, "display.max_rows", None):
        print(plan_df[["source_file", "col_question", "col_answer", "col_docs", "rows_in_csv", "rows_valid"]].to_string(index=False))
    print("\ncolumn names used:", {c: plan_df[c].value_counts().to_dict() for c in ("col_question", "col_answer", "col_docs")})
    for b in bad:
        print("  !!", b["source_file"], "->", b["error"])
    print(f"total rows to judge: {int(plan_df.rows_valid.sum())}  "
          f"(~{int(plan_df.rows_valid.sum()) * 3 * len(judge.labels)} judge calls in 'all' mode)\n")

    # 2) judges reachable?
    print("smoke test:")
    assert smoke_test(judge), "fix the judge models first"

    # 3) judge every file
    manifest = list(bad)
    for path, rel in tqdm(files, desc="files"):
        if any(b["source_file"] == rel for b in bad):
            continue
        try:
            manifest.append(process_file(judge, path, rel, CONTEXT_MODE, LIMIT_ROWS))
        except Exception as e:
            print(f"  !! {rel}: {type(e).__name__}: {e}")
            manifest.append({"source_file": rel, "status": f"ERROR {type(e).__name__}", "error": str(e)})
        pd.DataFrame(manifest).to_csv(os.path.join(OUT_DIR, "_manifest.csv"), index=False)

    build_outputs(judge)
    man = pd.DataFrame(manifest)
    print("\n=== DONE ===")
    print(man["status"].value_counts().to_dict())
    print("failed/unparsable calls per judge:", {l: f"{judge.n_failed[l]}/{judge.n_calls[l]}" for l in judge.labels})
    print("results folder:", OUT_DIR)
    return man


# ----------------------------------------------------------------------------- main
if globals().get("RUN_ALL", True):
    JUDGE = Judge(JUDGES, WEIGHTS)
    MANIFEST = run_all(JUDGE)